# CS781 Phase 7 — Prior Case Retrieval from the Phase 1 Legal LM, model 1 (PyTorch, Kaggle GPU)
Phase 1 JAX weights → PyTorch bi-encoder (chunked mean-pooled case embeddings) → contrastive fine-tuning with in-batch and TF-IDF hard negatives → LoRMA / IA³ / partial / full fine-tuning → distillation of similarity scores into a half-depth student → optional lexical fusion → top-7 per query, one `submission` zip per experiment.

## 1. Setup and configuration

In [1]:
import os, sys, time, json, math, random, pickle, zipfile, gc, copy, glob, subprocess, importlib.util
T0 = time.time()
if importlib.util.find_spec("regex") is None: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex"], check=False)
import numpy as np, regex as re2, torch, torch.nn as nn, torch.nn.functional as F
import multiprocessing as mp
from huggingface_hub import hf_hub_download, HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF"); login(token=HF_TOKEN)
except Exception as e:
    HF_TOKEN = None; print("HF login skipped:", e)

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
HF_REPO = "Exploration-Lab/CS-781-Capstone"; WORK = "/kaggle/working/cs781_phase7_m1"; os.makedirs(WORK, exist_ok=True)
PIN = None
FOUND = [(p, pickle.load(open(p, "rb"))) for p in glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)]
print("Phase 1 pickles found:", [(p, x["variant"]) for p, x in FOUND])
P1, d = next(((p, x) for p, x in FOUND if PIN in (None, p) and not any(k in x["cfg"] for k in ("kinds", "encdec", "adaln"))), (None, None)); del FOUND
assert d is not None, "no decoder-only ours_* final_params.pkl attached: add that Phase 1 run's output as a Kaggle dataset (or set PIN to its path)"
print("using", P1, "->", d["variant"])
TOKP = os.path.join(os.path.dirname(P1), "bpe_legal.json")
if not os.path.exists(TOKP): TOKP = glob.glob("/kaggle/input/**/bpe_legal.json", recursive=True)[0]
TOTAL_SEC, POST_SEC = 5.4 * 3600, 14 * 60
CT, KT, KI, BQ, HNN, TAU = 1024, 2, 8, 4, 2, 0.05
NG = torch.cuda.device_count()
EXPS = [("lorma", 1.0), ("ia3", 0.8), ("partial", 0.9), ("full", 1.1), ("kd", 1.2)]
LR = dict(lorma=3e-4, ia3=1e-3, partial=1e-4, full=3e-5, kd=5e-5)
print("gpus:", NG, "|", P1, "|", TOKP)

Phase 1 pickles found: [('/kaggle/input/datasets/nirbhaykachhatiya/phase2m1/final_params.pkl', 'ours_hybrid_gate')]
using /kaggle/input/datasets/nirbhaykachhatiya/phase2m1/final_params.pkl -> ours_hybrid_gate
gpus: 2 | /kaggle/input/datasets/nirbhaykachhatiya/phase2m1/final_params.pkl | /kaggle/input/datasets/nirbhaykachhatiya/phase2m1/bpe_legal.json


## 2. Data: Phase 4 CJPE files, Phase 1 byte-level BPE, tokenised once

In [2]:
PAT = re2.compile(r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+")
def b2u():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = b2u()

class BPE:
    def __init__(s, path):
        d = json.load(open(path, encoding="utf-8")); s.rank = {tuple(m): i for i, m in enumerate(d["merges"])}; s.stoi = {t: i for i, t in enumerate(d["itos"])}; s.c = {}
    def word(s, w):
        r = s.c.get(w)
        if r is None:
            x = [B2U[b] for b in w.encode("utf-8")]
            while len(x) > 1:
                best = min(zip(x, x[1:]), key=lambda q: s.rank.get(q, 1 << 60))
                if best not in s.rank: break
                a, b = best; y, i = [], 0
                while i < len(x):
                    if i < len(x) - 1 and x[i] == a and x[i + 1] == b: y.append(a + b); i += 2
                    else: y.append(x[i]); i += 1
                x = y
            r = s.c[w] = [s.stoi[t] for t in x]
        return r
    def encode(s, t): return [i for w in PAT.findall(t) for i in s.word(w)]

TK = BPE(TOKP)
from sklearn.feature_extraction.text import TfidfVectorizer
def norm(t): return " ".join(t.split())
def enc(ts): return [np.asarray(TK.encode(t), np.uint16) for t in ts]
def par(fn, xs):
    with mp.get_context("fork").Pool(os.cpu_count()) as pool: return [a for p in pool.imap(fn, [xs[i:i + 20] for i in range(0, len(xs), 20)]) for a in p]

files = HfApi().list_repo_files(HF_REPO, repo_type="dataset", token=HF_TOKEN)
def ld(name):
    fn = next(f for f in files if f.startswith("Phase_7_PCR/") and f.endswith("/" + name))
    return json.load(open(hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN, local_dir="/tmp/hf"), encoding="utf-8"))
def parse(obj):
    it = obj.items() if isinstance(obj, dict) else [(o.get("id", o.get("case_id", o.get("doc_id"))), o) for o in obj]; out = {}
    for k, v in it:
        t, rel = (v.get("text", v.get("document", "")), v.get("relevant_candidates")) if isinstance(v, dict) else (v, None)
        out[str(k)] = (norm(" ".join(map(str, t)) if isinstance(t, list) else str(t)), None if rel is None else sorted({str(r) for r in rel}))
    return out

TOKC = f"{WORK}/data.pkl"
if os.path.exists(TOKC): DS = pickle.load(open(TOKC, "rb"))
else:
    DS = {}; t0 = time.time()
    for sp in ("train", "dev", "test"):
        raw = ld(f"{sp}.json"); print(sp, "query file:", type(raw).__name__, "| first keys:", list(next(iter(raw.values())).keys()) if isinstance(raw, dict) and isinstance(next(iter(raw.values())), dict) else (list(raw[0].keys()) if isinstance(raw, list) else "n/a"))
        Q, Pl = parse(raw), parse(ld(f"{sp}_candidates.json")); qid, pid = list(Q), list(Pl); pidx = {p: i for i, p in enumerate(pid)}
        vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, max_features=300000, dtype=np.float32).fit([Pl[p][0] for p in pid])
        lex = (vec.transform([Q[q][0] for q in qid]) @ vec.transform([Pl[p][0] for p in pid]).T).toarray().astype(np.float32)
        toks = par(enc, [Q[q][0] for q in qid] + [Pl[p][0] for p in pid])
        gold = [[pidx[r] for r in Q[q][1] if r in pidx] if Q[q][1] is not None else None for q in qid]; gsz = [len(Q[q][1]) if Q[q][1] is not None else 0 for q in qid]
        DS[sp] = dict(qid=qid, pid=pid, pidx=pidx, qtok=toks[:len(qid)], ptok=toks[len(qid):], gold=gold, gsz=gsz, lex=lex)
        if sp == "train":
            DS[sp]["hn"] = [[int(j) for j in np.argsort(-lex[i])[:60] if j not in set(g)][:30] for i, g in enumerate(gold)]
        del Q, Pl, vec
    pickle.dump(DS, open(TOKC, "wb")); gc.collect(); print(f"prepared in {time.time()-t0:.0f}s")
for sp, x in DS.items(): print(f"{sp}: {len(x['qid'])} queries, pool {len(x['pid'])}, gold/query {np.mean(x['gsz']):.2f}, gold inside pool {np.mean([len(g) / max(s, 1) for g, s in zip(x['gold'], x['gsz'])]) if sp != 'test' else float('nan'):.2f}, median tokens q/pool {np.median([len(t) for t in x['qtok']]):.0f}/{np.median([len(t) for t in x['ptok']]):.0f}")

def selk(nc, rnd, K):
    if nc <= K: return list(range(nc))
    return sorted(random.sample(range(nc), K)) if rnd else np.unique(np.linspace(0, nc - 1, K).round().astype(int)).tolist()

def mkc(toks, K, rnd):
    ids = np.zeros((len(toks), K, CT), np.int64); ln = np.zeros((len(toks), K), np.int64)
    for b, t in enumerate(toks):
        n = len(t)
        for k, c in enumerate(selk(max(1, -(-n // CT)), rnd, K)):
            x = t[min(c * CT, max(n - CT, 0)):][:CT]; ids[b, k, :len(x)] = x; ln[b, k] = len(x)
    return torch.from_numpy(ids).cuda(), torch.from_numpy(ln).cuda()

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train query file: dict | first keys: ['text', 'relevant_candidates']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (4 outputs left out)

test query file: dict | first keys: ['text']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

prepared in 112s
train: 827 queries, pool 4320, gold/query 6.42, gold inside pool 0.99, median tokens q/pool 5805/4690
dev: 118 queries, pool 1023, gold/query 8.25, gold inside pool 1.00, median tokens q/pool 5830/5220
test: 237 queries, pool 1727, gold/query 0.00, gold inside pool nan, median tokens q/pool 6068/5043


## 3. Model: Phase 1 decoder ported to PyTorch, chunk encoder + attention pooling head, adaptation modules

In [3]:
d = pickle.load(open(P1, "rb")); P, C = d["params"], d["cfg"]; assert d["variant"].startswith("ours_"), d["variant"]
lp = P["layers"]; L, D, HHD = lp["wq"].shape; HD = lp["qn"].shape[1]; H = HHD // HD; KV = lp["wk"].shape[2] // HD; V = P["emb"].shape[0]; R = H // KV
WIN = 256; LOC = [bool(C["hybrid"]) and i % 3 != 2 for i in range(L)]; LINS = ("wq", "wk", "wv", "wo", "w1", "w3", "w2")
INV = torch.tensor(1.0 / (10000.0 ** (np.arange(0, HD, 2) / HD)), dtype=torch.float32)
print(d["variant"], f"L={L} D={D} H={H} KV={KV} HD={HD} V={V}", "|", torch.cuda.get_device_name(0))

def lin(w):
    m = nn.Linear(w.shape[0], w.shape[1], bias=False); m.weight.data = torch.from_numpy(np.ascontiguousarray(w.T, dtype=np.float32)); return m

class RMS(nn.Module):
    def __init__(s, w): super().__init__(); s.w = nn.Parameter(torch.from_numpy(np.asarray(w, np.float32).copy()))
    def forward(s, x): x = x.float(); return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * s.w

def rope(x, cos, sin):
    a, b = x[..., :HD // 2].float(), x[..., HD // 2:].float()
    return torch.cat([a * cos - b * sin, b * cos + a * sin], -1)

class Blk(nn.Module):
    def __init__(s, i):
        super().__init__()
        for n in LINS: setattr(s, n, lin(lp[n][i]))
        s.n1, s.n2, s.qn, s.kn = RMS(lp["n1"][i]), RMS(lp["n2"][i]), RMS(lp["qn"][i]), RMS(lp["kn"][i])
        s.g = nn.Parameter(torch.from_numpy(np.asarray(lp["g"][i], np.float32).copy())) if "g" in lp else None
    def gate(s, j): return 1.0 if s.g is None else s.g[j]
    def forward(s, x, cos, sin, mask):
        B, T, _ = x.shape; h = s.n1(x)
        q, k, v = s.wq(h).view(B, T, H, HD), s.wk(h).view(B, T, KV, HD), s.wv(h).view(B, T, KV, HD)
        q, k = rope(s.qn(q), cos, sin).to(v.dtype), rope(s.kn(k), cos, sin).to(v.dtype)
        o = F.scaled_dot_product_attention(q.transpose(1, 2), k.repeat_interleave(R, 2).transpose(1, 2), v.repeat_interleave(R, 2).transpose(1, 2), attn_mask=mask, is_causal=mask is None)
        x = x + s.gate(0) * s.wo(o.transpose(1, 2).reshape(B, T, H * HD)); h = s.n2(x)
        return x + s.gate(1) * s.w2(F.silu(s.w1(h)) * s.w3(h))

class Enc(nn.Module):
    def __init__(s, keep):
        super().__init__(); s.emb = nn.Embedding(V, D); s.emb.weight.data = torch.from_numpy(np.asarray(P["emb"], np.float32).copy())
        s.blocks = nn.ModuleList([Blk(i) for i in keep]); s.nf = RMS(P["nf"]); s.loc = [LOC[i] for i in keep]
        t = torch.arange(CT)[:, None] * INV[None]; s.register_buffer("cos", t.cos()[None, :, None], persistent=False); s.register_buffer("sin", t.sin()[None, :, None], persistent=False)
    def forward(s, ids):
        T = ids.shape[1]; x = s.emb(ids); cos, sin = s.cos[:, :T], s.sin[:, :T]
        loc = torch.ones(T, T, dtype=torch.bool, device=ids.device).tril().triu(1 - WIN)
        for b, l in zip(s.blocks, s.loc): x = b(x, cos, sin, loc if l else None)
        return s.nf(x)

class DocEnc(nn.Module):
    def __init__(s, enc): super().__init__(); s.enc = enc
    def forward(s, ids, ln):
        B, Kc, T = ids.shape; h = s.enc(ids.view(B * Kc, T)).float().view(B, Kc, T, -1)
        tm = (torch.arange(T, device=ids.device)[None, None] < ln[..., None]).float(); hc = (h * tm[..., None]).sum(2) / ln.clamp(min=1)[..., None]; cm = (ln > 0).float()[..., None]
        return F.normalize((hc * cm).sum(1) / cm.sum(1).clamp(min=1), dim=-1)

class LoRMA(nn.Module):
    def __init__(s, b, r=16): super().__init__(); s.b = b; o = b.out_features; s.A = nn.Parameter(torch.randn(o, r) * o ** -0.5); s.B = nn.Parameter(torch.zeros(r, o))
    def forward(s, x): y = s.b(x); return y + (y @ s.A) @ s.B

class Scale(nn.Module):
    def __init__(s, b): super().__init__(); s.b = b; s.l = nn.Parameter(torch.ones(b.out_features))
    def forward(s, x): y = s.b(x); return y * s.l.to(y.dtype)

class ScaleIn(nn.Module):
    def __init__(s, b): super().__init__(); s.b = b; s.l = nn.Parameter(torch.ones(b.in_features))
    def forward(s, x): return s.b(x * s.l)

def adapt(m, kind):
    if kind in ("full", "kd"): return m
    for p in m.parameters(): p.requires_grad = False
    if kind == "partial":
        for p in [*m.enc.nf.parameters(), *[q for b in m.enc.blocks[-4:] for q in b.parameters()]]: p.requires_grad = True
        return m
    for b in m.enc.blocks:
        if kind == "lorma":
            for n in LINS: setattr(b, n, LoRMA(getattr(b, n)))
        elif kind == "ia3": b.wk, b.wv, b.w2 = Scale(b.wk), Scale(b.wv), ScaleIn(b.w2)
    return m

def build(kind): return adapt(DocEnc(Enc(range(0, L, 2) if kind == "kd" else range(L))), kind)

ours_hybrid_gate L=12 D=768 H=12 KV=4 HD=64 V=16384 | Tesla T4


## 4. Training utilities (DataParallel, fp16 autocast, class-weighted loss, time-budgeted cosine schedule)

In [4]:
LAMS = (0.0, 0.25, 0.5, 1.0, 2.0)
def zs(S): return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
def retrieve(sp, Eq, Ep, lam):
    x = DS[sp]; S = zs(x["lex"]) * lam
    if Eq is not None: S = S + zs((Eq @ Ep.T).numpy())
    for i, q in enumerate(x["qid"]):
        j = x["pidx"].get(q)
        if j is not None: S[i, j] = -1e9
    return np.argsort(-S, 1)[:, :7]
def f7(sp, top):
    x = DS[sp]; tp = sum(len(set(t.tolist()) & set(g)) for t, g in zip(top, x["gold"])); P_, R_ = tp / (7 * len(top)), tp / max(sum(x["gsz"]), 1)
    return dict(f1=2 * P_ * R_ / (P_ + R_) if tp else 0.0, p=P_, r=R_)
def best_lam(sp, Eq, Ep):
    tab = {l: f7(sp, retrieve(sp, Eq, Ep, l)) for l in LAMS}; l = max(tab, key=lambda k: tab[k]["f1"]); return l, tab

@torch.no_grad()
def embed(dm, toks, K=None, bs=16):
    K = K or KI; tr = dm.training; dm.eval(); out = []
    for i in range(0, len(toks), bs):
        ids, ln = mkc(toks[i:i + bs], K, False)
        with torch.autocast("cuda", torch.float16): out.append(dm(ids, ln).float().cpu())
    dm.train(tr); return torch.cat(out)

def embed_split(dm, sp, K=None): return embed(dm, DS[sp]["qtok"], K), embed(dm, DS[sp]["ptok"], K)

def fit(name, m, kind, budget, teacher=None):
    m.cuda().train(); dm = nn.DataParallel(m) if NG > 1 else m; tdm = (nn.DataParallel(teacher) if NG > 1 else teacher) if teacher is not None else None
    tp = [p for p in m.parameters() if p.requires_grad]; opt = torch.optim.AdamW(tp, lr=LR[kind], weight_decay=0.01, betas=(0.9, 0.98)); sc = torch.amp.GradScaler("cuda")
    tr = DS["train"]; ok = [i for i, g in enumerate(tr["gold"]) if g and len(tr["hn"][i]) >= HNN]; labels = torch.arange(BQ).cuda()
    step, total, tb, ema, hist, t0 = 0, 10 ** 9, None, None, [], time.time(); torch.cuda.reset_peak_memory_stats()
    while step < total and time.time() - T0 < TOTAL_SEC - POST_SEC:
        qs = random.sample(ok, BQ); qt, pt, nt, did = [], [], [], []
        for qi in qs: p = random.choice(tr["gold"][qi]); did.append(p); qt.append(tr["qtok"][qi]); pt.append(tr["ptok"][p])
        for qi in qs:
            for j in random.sample(tr["hn"][qi], HNN): did.append(j); nt.append(tr["ptok"][j])
        gs = [set(tr["gold"][qi]) for qi in qs]; mask = torch.tensor([[(did[j] in gs[i]) and j != i for j in range(len(did))] for i in range(BQ)]).cuda()
        ids, ln = mkc(qt + pt + nt, KT, True)
        f = min(1.0, (step + 1) / 50) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(step / total, 1.0))))
        for g in opt.param_groups: g["lr"] = LR[kind] * f
        with torch.autocast("cuda", torch.float16):
            E = dm(ids, ln).float()
            if tdm is not None:
                with torch.no_grad(): Et = tdm(ids, ln).float()
        S = (E[:BQ] @ E[BQ:].T / TAU).masked_fill(mask, -1e4); loss = F.cross_entropy(S, labels)
        if tdm is not None: St = (Et[:BQ] @ Et[BQ:].T / TAU).masked_fill(mask, -1e4); loss = 0.5 * loss + 0.5 * F.kl_div(F.log_softmax(S, -1), F.softmax(St, -1), reduction="batchmean")
        opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.unscale_(opt); nn.utils.clip_grad_norm_(tp, 1.0); sc.step(opt); sc.update(); step += 1
        if total == 10 ** 9:
            if step == 5: torch.cuda.synchronize(); tb = time.time()
            if step == 25:
                torch.cuda.synchronize(); rate = (time.time() - tb) / 20; total = step + int(0.97 * (budget - (time.time() - t0)) / rate)
                print(f"[{name}] {rate:.3f}s/step -> total_steps={total} (~{total * BQ / len(ok):.1f} epochs over {len(ok)} queries)", flush=True)
        if step % 10 == 0: ema = loss.item() if ema is None else 0.9 * ema + 0.1 * loss.item()
        if step % 50 == 0 and ema is not None: hist.append((step, ema))
        if step % 200 == 0: print(f"[{name}] step {step}/{total} loss {ema:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
    tt = time.time() - t0; mem = torch.cuda.max_memory_allocated() / 2 ** 30; del opt, sc; gc.collect(); torch.cuda.empty_cache(); m.eval()
    t1 = time.time(); Eq, Ep = embed_split(dm, "dev"); lam, tab = best_lam("dev", Eq, Ep); es = time.time() - t1; Eqt, Ept = embed_split(dm, "test")
    top = retrieve("test", Eqt, Ept, lam); out = {DS["test"]["qid"][i]: [DS["test"]["pid"][j] for j in t] for i, t in enumerate(top)}
    assert all(len(set(v)) == 7 for v in out.values()) and len(out) == len(DS["test"]["qid"])
    os.makedirs(f"{WORK}/{name}", exist_ok=True); pj = f"{WORK}/{name}/prediction.json"; json.dump(out, open(pj, "w"))
    with zipfile.ZipFile(f"{WORK}/sub_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    np.save(f"{WORK}/{name}/dev_q.npy", Eq.half().numpy()); np.save(f"{WORK}/{name}/dev_p.npy", Ep.half().numpy()); torch.save({n: p.detach().cpu() for n, p in m.named_parameters() if p.requires_grad}, f"{WORK}/{name}/weights.pt")
    res = dict(name=name, f1=tab[lam]["f1"], p=tab[lam]["p"], r=tab[lam]["r"], lam=lam, f1_dense=tab[0.0]["f1"], lam_table={str(k): v["f1"] for k, v in tab.items()}, params_M=sum(p.numel() for p in m.parameters()) / 1e6,
               train_M=sum(p.numel() for p in tp) / 1e6, steps=step, train_min=tt / 60, mem_GB=mem, embed_s=es, hist=hist)
    print(f"[{name}] dev F1@7 {res['f1']:.4f} (P {res['p']:.4f} R {res['r']:.4f}) at lambda={lam} | dense-only {res['f1_dense']:.4f} | embed dev {es:.0f}s", flush=True); return res, m

## 5. Experiments (time-shared budget; KD teacher = best dev macro-F1 model so far, student = every-other-layer Phase 1 init)

In [5]:
z = DocEnc(Enc(range(L))).cuda().eval(); zdm = nn.DataParallel(z) if NG > 1 else z; Eq, Ep = embed_split(zdm, "dev"); zl, ztab = best_lam("dev", Eq, Ep); lex = f7("dev", retrieve("dev", None, None, 1.0))
print(f"BASELINES on dev (F1@7): TF-IDF only {lex['f1']:.4f} | Phase 1 zero-shot dense {ztab[0.0]['f1']:.4f} | zero-shot + TF-IDF fusion {ztab[zl]['f1']:.4f} (lambda {zl})", flush=True)
ZS = dict(tfidf=lex["f1"], dense=ztab[0.0]["f1"], fused=ztab[zl]["f1"]); json.dump(ZS, open(f"{WORK}/baselines.json", "w")); del z, zdm, Eq, Ep; gc.collect(); torch.cuda.empty_cache()
RES, BEST = {}, (-1.0, None)
for i, (name, w) in enumerate(EXPS):
    alloc = (TOTAL_SEC - (time.time() - T0)) * w / sum(x[1] for x in EXPS[i:])
    if alloc < 25 * 60: print("skip", name, "(not enough time)"); continue
    print(f"=== {name}: {alloc / 60:.0f} min allocated, {(time.time() - T0) / 60:.0f} min elapsed", flush=True)
    assert name != "kd" or BEST[1] is not None
    tc = BEST[1].cuda().eval() if name == "kd" else None
    if tc is not None:
        for p in tc.parameters(): p.requires_grad = False
    RES[name], m = fit(name, build(name), name, alloc - POST_SEC, tc)
    if name != "kd" and RES[name]["f1"] > BEST[0]: BEST = (RES[name]["f1"], copy.deepcopy(m).cpu())
    if tc is not None: tc.cpu()
    m.cpu(); del m, tc; gc.collect(); torch.cuda.empty_cache(); json.dump(RES, open(f"{WORK}/results.json", "w"), indent=1)

print(f"{'exp':9s}{'total M':>9s}{'train M':>9s}{'F1@7':>8s}{'P@7':>8s}{'R@7':>8s}{'lam':>6s}{'dense F1':>10s}{'min':>7s}{'GB':>6s}{'embed s':>9s}")
for k, r in RES.items(): print(f"{k:9s}{r['params_M']:9.1f}{r['train_M']:9.2f}{r['f1']:8.4f}{r['p']:8.4f}{r['r']:8.4f}{r['lam']:6.2f}{r['f1_dense']:10.4f}{r['train_min']:7.1f}{r['mem_GB']:6.1f}{r['embed_s']:9.0f}")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for k, r in RES.items():
    if r["hist"]: plt.plot(*zip(*r["hist"]), label=k)
plt.xlabel("step"); plt.ylabel("train loss (EMA)"); plt.grid(True); plt.legend(); plt.savefig(f"{WORK}/loss_curves.png", dpi=200, bbox_inches="tight"); plt.show()
print("submissions:", sorted(glob.glob(f"{WORK}/sub_*.zip")))

BASELINES on dev (F1@7): TF-IDF only 0.3558 | Phase 1 zero-shot dense 0.2490 | zero-shot + TF-IDF fusion 0.3613 (lambda 2.0)
=== lorma: 64 min allocated, 4 min elapsed
[lorma] 1.254s/step -> total_steps=2324 (~11.4 epochs over 817 queries)
[lorma] step 200/2324 loss 1.7273 | 4.3 min
[lorma] step 400/2324 loss 1.8180 | 8.7 min
[lorma] step 600/2324 loss 1.4868 | 13.0 min
[lorma] step 800/2324 loss 1.8022 | 17.4 min
[lorma] step 1000/2324 loss 1.5234 | 21.8 min
[lorma] step 1200/2324 loss 1.7055 | 26.2 min
[lorma] step 1400/2324 loss 1.4477 | 30.6 min
[lorma] step 1600/2324 loss 1.1744 | 34.9 min
[lorma] step 1800/2324 loss 1.2887 | 39.3 min
[lorma] step 2000/2324 loss 1.2023 | 43.6 min
[lorma] step 2200/2324 loss 1.1405 | 48.0 min
[lorma] dev F1@7 0.3958 (P 0.4310 R 0.3659) at lambda=0.5 | dense-only 0.3346 | embed dev 128s
=== ia3: 53 min allocated, 60 min elapsed
[ia3] 1.214s/step -> total_steps=1857 (~9.1 epochs over 817 queries)
[ia3] step 200/1857 loss 1.8659 | 4.0 min
[ia3] step 4

## 6. Long-document analysis: which part of the judgment carries the signal (best non-KD model, dev set)

In [6]:
bm = BEST[1].cuda().eval(); dm = nn.DataParallel(bm) if NG > 1 else bm
print("pooling over chunks: dev F1@7 (dense only / best fusion)")
for K_ in (1, 4, KI):
    Eq, Ep = embed_split(dm, "dev", K_); l, tab = best_lam("dev", Eq, Ep); print(f"  {K_} chunks/doc: {tab[0.0]['f1']:.4f} / {tab[l]['f1']:.4f} (lambda {l})")
Eq, Ep = embed_split(dm, "dev"); l, tab = best_lam("dev", Eq, Ep); print("lambda sweep:", {k: round(v["f1"], 4) for k, v in tab.items()})
top = retrieve("dev", Eq, Ep, l); x = DS["dev"]; hits = np.array([len(set(t.tolist()) & set(g)) for t, g in zip(top, x["gold"])]); gs = np.array(x["gsz"]); ql = np.array([len(t) for t in x["qtok"]])
print("queries by hits@7:", np.bincount(hits, minlength=8).tolist(), "| zero-hit share", round(float((hits == 0).mean()), 3))
for lo, hi in ((0, 3), (3, 6), (6, 10), (10, 10 ** 6)):
    s_ = (gs >= lo) & (gs < hi)
    if s_.any(): print(f"  gold size [{lo},{hi}): {int(s_.sum())} queries, recall@7 {hits[s_].sum() / max(gs[s_].sum(), 1):.3f}")
for nm, s_ in (("short queries (<= median length)", ql <= np.median(ql)), ("long queries (> median length)", ql > np.median(ql))): print(f"  {nm}: precision@7 {hits[s_].sum() / (7 * max(int(s_.sum()), 1)):.3f}")
bm.cpu()

pooling over chunks: dev F1@7 (dense only / best fusion)
  1 chunks/doc: 0.2779 / 0.3802 (lambda 1.0)
  4 chunks/doc: 0.3380 / 0.3980 (lambda 0.5)
  8 chunks/doc: 0.3402 / 0.3969 (lambda 1.0)
lambda sweep: {0.0: 0.3402, 0.25: 0.388, 0.5: 0.3958, 1.0: 0.3969, 2.0: 0.3847}
queries by hits@7: [9, 24, 21, 20, 13, 14, 10, 7] | zero-hit share 0.076
  gold size [0,3): 33 queries, recall@7 0.725
  gold size [3,6): 30 queries, recall@7 0.583
  gold size [6,10): 25 queries, recall@7 0.566
  gold size [10,1000000): 30 queries, recall@7 0.233
  short queries (<= median length): precision@7 0.337
  long queries (> median length): precision@7 0.528


DocEnc(
  (enc): Enc(
    (emb): Embedding(16384, 768)
    (blocks): ModuleList(
      (0-11): 12 x Blk(
        (wq): Linear(in_features=768, out_features=768, bias=False)
        (wk): Linear(in_features=768, out_features=256, bias=False)
        (wv): Linear(in_features=768, out_features=256, bias=False)
        (wo): Linear(in_features=768, out_features=768, bias=False)
        (w1): Linear(in_features=768, out_features=2048, bias=False)
        (w3): Linear(in_features=768, out_features=2048, bias=False)
        (w2): Linear(in_features=2048, out_features=768, bias=False)
        (n1): RMS()
        (n2): RMS()
        (qn): RMS()
        (kn): RMS()
      )
    )
    (nf): RMS()
  )
)